In [ ]:
!pip install llama-index llama-index-multi-modal-llms-openai

INFO: pip is looking at multiple versions of llama-index-multi-modal-llms-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-index-multi-modal-llms-openai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of llama-index-cli to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version 

In [ ]:
questions = ['ماهي احصائيات الاشخاص المعاقين في عام 2016 ضمن محافظة الظاهرة ؟' ,
             'من الجدول رقم (2) ماهي الفئة العمرية التي تحتوي على اقل عدد من الاشخاص ذوي الاعاقة ؟',
             'من الجدول رقم (4) ماهي الاحصائيات المتعلقة بمرض المهق لسنة 2017 ؟' ]

# GPT-5.1

In [ ]:
# @title Step 1 – Extraction

import os
import base64
from openai import OpenAI as RawOpenAI
from llama_index.llms.openai import OpenAI
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core.base.llms.types import TextBlock
from llama_index.llms.openai.utils import ALL_AVAILABLE_MODELS, CHAT_MODELS

os.environ["OPENROUTER_API_KEY"] = 'your key'


pdf_path = "page_1.pdf"
with open(pdf_path, "rb") as f:
    pdf_base64 = base64.standard_b64encode(f.read()).decode("utf-8")


# ── Step 1 – OCR using raw OpenAI client (bypasses LlamaIndex validation) ───

raw_client = RawOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

prompt = (
    "Extract the text from the above document as if you were reading it naturally. "
    "Return the tables in html format. "
    "Return the equations in LaTeX representation. "
    "If there is an image in the document wrap it as follows <img>image here</img>. "
    "Watermarks should be wrapped in brackets. Ex: <watermark>OFFICIAL COPY</watermark>. "
    "Page numbers should be wrapped in brackets. Ex: <page_number>14</page_number>."
)


OCR_MODEL_ID = "openai/gpt-5.1"

response = raw_client.chat.completions.create(
    model= OCR_MODEL_ID,
    max_tokens=16000,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "file",
                    "file": {
                        "filename": "document.pdf",
                        "file_data": f"data:application/pdf;base64,{pdf_base64}",
                    },
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ],
)

OCR_response = response.choices[0].message.content

print("OCR done ✓")
print('-'*20)
print(OCR_response)
print('─'*80)

with open('GPT-5_1.txt', 'w') as f:
  f.write(OCR_response)

OCR done ✓
--------------------
<page_number>6</page_number>

الأشخاص ذوو الإعاقة  

المركز الوطني للإحصاء والمعلومات  
سلطنة عُمان  

سلسلة الإحصاءات المجتمعية  
(إصدار 2018م)  

---

<img>image here</img>

---

<page_number>11</page_number>

1- التوزيع الجغرافي للأشخاص ذوي الإعاقة  

يتضح من الجدول رقم (1) أن عدد الأشخاص المسجلين بنظام بطاقة شخص معاقَة المسلمة ارتفع من  
(24,249) عام 2016م ليصل إلى (25,325) معاقًا في عام 2017م، بشكل الذكور ما نسبته (65%) والإناث  
(35%). كما ارتفعت النسبة لمعظم المحافظات غير أن أعلى نسبة سجلت منهم في محافظة شمال الباطنة، حيث  
شكلوا في عام 2017م ما نسبته (26%) من إجماليهم على مستوى المحافظات، تلتها محافظة مسقط (21.6%)،  
بينما سجلت أدنى نسبة في محافظتي جنوب الداخلية وأدمًا في محافظة الوسطى.  

ملاحظة:  
الإعاقة هي حدوث خلل كلي أو جزئي دائم أو مؤقت في بعض قدراتك الحسية أو الجسدية أو  
العقلية أو النفسية، مما يقلل من إمكانية تلبية متطلبات حياتك العادية في بيئتك الطبيعية، من  
خلال الاعتماد على نفسك أو بالاستعانة بالأجهزة المساعدة أو مساعدة الغير أو الا

In [ ]:
# @title Step 2 – Answer question with LLM-MODEL via LlamaIndex

OCR_response = open('GPT-5_1.txt', 'r').read()

LLM_ID = "google/gemini-2.5-flash"
ALL_AVAILABLE_MODELS[LLM_ID] = 1048576
CHAT_MODELS[LLM_ID] = 1048576

LLM_MODEL = OpenAI(
    model=LLM_ID,
    api_key=os.environ["OPENROUTER_API_KEY"],
    api_base="https://openrouter.ai/api/v1",
    max_tokens=16000,
)

# question = "كم عدد الذكور المعاقين من محافظة جنوب الشرقية في عام 2016"

for question in questions:
  LLM_prompt = f"""تم تزويدك بالنص المستخرج التالي من أحد المستندات:

---
{OCR_response}
---

أجب الآن عن هذا السؤال بناءً على النص المستخرج أعلاه:
{question}
"""

  messages = [
      ChatMessage(
          role=MessageRole.USER,
          blocks=[TextBlock(text=LLM_prompt)],
      )
  ]

  LLM_response = LLM_MODEL.chat(messages)
  print('─'*80)
  print(question)
  print('-'*20)
  print(LLM_response.message.content)
  print('─'*80)

────────────────────────────────────────────────────────────────────────────────
ماهي احصائيات الاشخاص المعاقين في عام 2016 ضمن محافظة الظاهرة ؟
--------------------
في عام 2016، كانت إحصائيات الأشخاص المعاقين في محافظة الظاهرة كالتالي:

*   **ذكور:** 1,233
*   **إناث:** 594
*   **المجموع:** 1,827
────────────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────────────
من الجدول رقم (2) ماهي الفئة العمرية التي تحتوي على اقل عدد من الاشخاص ذوي الاعاقة ؟
--------------------
من الجدول رقم (2)، الفئة العمرية التي تحتوي على أقل عدد من الأشخاص ذوي الإعاقة هي **(15-17)** في عام 2016م بمجموع 3,240 شخصًا، وفي عام 2017م بمجموع 3,281 شخصًا.
────────────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────────────
من الجدول رقم (4) ماهي الاحصائيات المتعلقة بمرض المهق لسنة 2017 ؟
--------------------
النص المستخرج لا يحتوي 

# GPT-4.1

In [ ]:
# @title Step 1 – Extraction

import os
import base64
from openai import OpenAI as RawOpenAI
from llama_index.llms.openai import OpenAI
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core.base.llms.types import TextBlock
from llama_index.llms.openai.utils import ALL_AVAILABLE_MODELS, CHAT_MODELS

os.environ["OPENROUTER_API_KEY"] = 'your key'


pdf_path = "page_1.pdf"
with open(pdf_path, "rb") as f:
    pdf_base64 = base64.standard_b64encode(f.read()).decode("utf-8")


# ── Step 1 – OCR using raw OpenAI client (bypasses LlamaIndex validation) ───

raw_client = RawOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

prompt = (
    "Extract the text from the above document as if you were reading it naturally. "
    "Return the tables in html format. "
    "Return the equations in LaTeX representation. "
    "If there is an image in the document wrap it as follows <img>image here</img>. "
    "Watermarks should be wrapped in brackets. Ex: <watermark>OFFICIAL COPY</watermark>. "
    "Page numbers should be wrapped in brackets. Ex: <page_number>14</page_number>."
)


OCR_MODEL_ID = "openai/gpt-4.1"

response = raw_client.chat.completions.create(
    model= OCR_MODEL_ID,
    max_tokens=16000,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "file",
                    "file": {
                        "filename": "document.pdf",
                        "file_data": f"data:application/pdf;base64,{pdf_base64}",
                    },
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ],
)

OCR_response = response.choices[0].message.content

print("OCR done ✓")
print('-'*20)
print(OCR_response)
print('─'*80)

with open('GPT-4_1.txt', 'w') as f:
  f.write(OCR_response)

OCR done ✓
--------------------
Certainly! Here is the extracted text from the document as if you were reading it naturally, with the requested formatting for tables, images, watermarks, and page numbers.

---

<page_number>6</page_number>

<img>  
[Cover page image]
</img>

الأشخاص ذوو الإعاقة  
سلسلة الإحصاءات المجتمعية  
إصدار 2018  
المركز الوطني للإحصاء والمعلومات  
سلطنة عمان  

---

<img>
[Demographic situation section page image]
</img>

الوضع الديموغرافي

---

<page_number>11</page_number>

**التوزيع الجغرافي للأشخاص ذوي الإعاقة**

يتضح من الجدول رقم (1) أن عدد الأشخاص المسجلين بنظام بطاقة شخص معاقًة المسلمة للمستفيد من 2016 إلى 2017م ليصل إلى (34,265) معاقًا في عام 2017م، شكل الذكور ما نسبته (65.7%) والإناث (34.3%).  
كما الملاحظ بالنسبة للمحافظات فقد تركزت أعلى نسبة منهم في محافظة شمال الباطنة، حيث شكلوا في عام 2017م ما نسبته (20.3%) من إجماليهم على مستوى المحافظات، تلاها (20.1%) ذكور، مقابل (20.7%) إناث، ثم باقي محافظات السلطنة الداخلية وأدناها في محافظة الوسطى.

**إعاقة**:

In [ ]:
# @title Step 2 – Answer question with LLM-MODEL via LlamaIndex

OCR_response = open('GPT-4_1.txt', 'r').read()

LLM_ID = "google/gemini-2.5-flash"
ALL_AVAILABLE_MODELS[LLM_ID] = 1048576
CHAT_MODELS[LLM_ID] = 1048576

LLM_MODEL = OpenAI(
    model=LLM_ID,
    api_key=os.environ["OPENROUTER_API_KEY"],
    api_base="https://openrouter.ai/api/v1",
    max_tokens=16000,
)

# question = "كم عدد الذكور المعاقين من محافظة جنوب الشرقية في عام 2016"

for question in questions:
  LLM_prompt = f"""تم تزويدك بالنص المستخرج التالي من أحد المستندات:

---
{OCR_response}
---

أجب الآن عن هذا السؤال بناءً على النص المستخرج أعلاه:
{question}
"""

  messages = [
      ChatMessage(
          role=MessageRole.USER,
          blocks=[TextBlock(text=LLM_prompt)],
      )
  ]

  LLM_response = LLM_MODEL.chat(messages)
  print('─'*80)
  print(question)
  print('-'*20)
  print(LLM_response.message.content)
  print('─'*80)

────────────────────────────────────────────────────────────────────────────────
ماهي احصائيات الاشخاص المعاقين في عام 2016 ضمن محافظة الظاهرة ؟
--------------------
بالنظر إلى **جدول (1): التوزيع العددي للمسجلين بنظام بطاقة شخص معاق حسب النوع والمحافظات لعامي (2016 - 2017م)**، نجد أن إحصائيات الأشخاص ذوي الإعاقة في محافظة الظاهرة لعام 2016 هي كالتالي:

*   **ذكور:** 1,067
*   **إناث:** 602
*   **المجموع:** 1,669
────────────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────────────
من الجدول رقم (2) ماهي الفئة العمرية التي تحتوي على اقل عدد من الاشخاص ذوي الاعاقة ؟
--------------------
من الجدول رقم (2)، الفئة العمرية التي تحتوي على أقل عدد من الأشخاص ذوي الإعاقة هي **(15-29)**.

*   في عام 2016، كان مجموع هذه الفئة 5,296 شخصًا.
*   في عام 2017، كان مجموع هذه الفئة 5,338 شخصًا.

بينما الفئات الأخرى كانت:
*   (أقل من 15): 7,744 شخصًا في كلا العامين.
*   (30 فأكثر): 21,022 شخصًا في 2016 و 21,683 ش

# GPT-4o-mini

In [ ]:
# @title Step 1 – Extraction

import os
import base64
from openai import OpenAI as RawOpenAI
from llama_index.llms.openai import OpenAI
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core.base.llms.types import TextBlock
from llama_index.llms.openai.utils import ALL_AVAILABLE_MODELS, CHAT_MODELS

os.environ["OPENROUTER_API_KEY"] = 'your key'


pdf_path = "page_1.pdf"
with open(pdf_path, "rb") as f:
    pdf_base64 = base64.standard_b64encode(f.read()).decode("utf-8")


# ── Step 1 – OCR using raw OpenAI client (bypasses LlamaIndex validation) ───

raw_client = RawOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

prompt = (
    "Extract the text from the above document as if you were reading it naturally. "
    "Return the tables in html format. "
    "Return the equations in LaTeX representation. "
    "If there is an image in the document wrap it as follows <img>image here</img>. "
    "Watermarks should be wrapped in brackets. Ex: <watermark>OFFICIAL COPY</watermark>. "
    "Page numbers should be wrapped in brackets. Ex: <page_number>14</page_number>."
)


OCR_MODEL_ID = 'openai/gpt-4o-mini'

response = raw_client.chat.completions.create(
    model= OCR_MODEL_ID,
    max_tokens=16000,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "file",
                    "file": {
                        "filename": "document.pdf",
                        "file_data": f"data:application/pdf;base64,{pdf_base64}",
                    },
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ],
)

OCR_response = response.choices[0].message.content

print("OCR done ✓")
print('-'*20)
print(OCR_response)
print('─'*80)

with open('GPT-4o_mini.txt', 'w') as f:
  f.write(OCR_response)

OCR done ✓
--------------------
Sure! Here's the text extracted naturally from the document, with tables in HTML format and equations in LaTeX representation. I've wrapped images and watermarks as specified.

---

**Page 1**:
[<watermark>OFFICIAL COPY</watermark>]
**الأشخاص ذوو الإعاقة**
سلسلة الإحصاءات المجتمعية (إصدار 2018)

---

**Page 2**:
[<watermark>OFFICIAL COPY</watermark>]
**الوضع الديموغرافي**

---

**Page 3**:
[<watermark>OFFICIAL COPY</watermark>]

### 1- التوزيع الجغرافي للأشخاص ذوي الإعاقة

<table>
    <tr>
        <th>المنطقة</th>
        <th>النسبة المئوية</th>
    </tr>
    <tr>
        <td>المنطقة 1</td>
        <td>25.0%</td>
    </tr>
    <tr>
        <td>المنطقة 2</td>
        <td>30.0%</td>
    </tr>
    <tr>
        <td>المنطقة 3</td>
        <td>45.0%</td>
    </tr>
</table>

---

**Page 4**:
[<watermark>OFFICIAL COPY</watermark>]

### 3- التوزيع النوعي والعُمري للأشخاص ذوي الإعاقة

<table>
    <tr>
        <th>النوع</th>
        <th>الفئة العمرية</th>
        <

In [ ]:
# @title Step 2 – Answer question with LLM-MODEL via LlamaIndex

OCR_response = open('GPT-4o_mini.txt', 'r').read()

LLM_ID = "google/gemini-2.5-flash"
ALL_AVAILABLE_MODELS[LLM_ID] = 1048576
CHAT_MODELS[LLM_ID] = 1048576

LLM_MODEL = OpenAI(
    model=LLM_ID,
    api_key=os.environ["OPENROUTER_API_KEY"],
    api_base="https://openrouter.ai/api/v1",
    max_tokens=16000,
)

# question = "كم عدد الذكور المعاقين من محافظة جنوب الشرقية في عام 2016"

for question in questions:
  LLM_prompt = f"""تم تزويدك بالنص المستخرج التالي من أحد المستندات:

---
{OCR_response}
---

أجب الآن عن هذا السؤال بناءً على النص المستخرج أعلاه:
{question}
"""

  messages = [
      ChatMessage(
          role=MessageRole.USER,
          blocks=[TextBlock(text=LLM_prompt)],
      )
  ]

  LLM_response = LLM_MODEL.chat(messages)
  print('─'*80)
  print(question)
  print('-'*20)
  print(LLM_response.message.content)
  print('─'*80)

────────────────────────────────────────────────────────────────────────────────
ماهي احصائيات الاشخاص المعاقين في عام 2016 ضمن محافظة الظاهرة ؟
--------------------
النص المستخرج لا يحتوي على أي معلومات عن إحصائيات الأشخاص ذوي الإعاقة في عام 2016 ضمن محافظة الظاهرة. المعلومات المتاحة هي عن "سلسلة الإحصاءات المجتمعية (إصدار 2018)" وتتضمن توزيعات جغرافية ونوعية وعمرية وأنواع إعاقات، ولكن لا يوجد تفصيل حسب المحافظات أو سنوات محددة غير 2018.
────────────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────────────
من الجدول رقم (2) ماهي الفئة العمرية التي تحتوي على اقل عدد من الاشخاص ذوي الاعاقة ؟
--------------------
لا يمكن الإجابة على هذا السؤال بناءً على النص المستخرج.

الجدول المشار إليه في السؤال هو "3- التوزيع النوعي والعُمري للأشخاص ذوي الإعاقة" (في الصفحة 4). هذا الجدول يعرض النسب المئوية للأشخاص ذوي الإعاقة حسب النوع والفئة العمرية، ولكنه لا يقدم معلومات عن "عدد الأشخاص" أو "أقل عدد" بشكل مبا

# Gemini-3-flash-preview

In [ ]:
# @title Step 1 – Extraction

import os
import base64
from openai import OpenAI as RawOpenAI
from llama_index.llms.openai import OpenAI
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core.base.llms.types import TextBlock
from llama_index.llms.openai.utils import ALL_AVAILABLE_MODELS, CHAT_MODELS

os.environ["OPENROUTER_API_KEY"] = 'your key'


pdf_path = "page_1.pdf"
with open(pdf_path, "rb") as f:
    pdf_base64 = base64.standard_b64encode(f.read()).decode("utf-8")


# ── Step 1 – OCR using raw OpenAI client (bypasses LlamaIndex validation) ───

raw_client = RawOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

prompt = (
    "Extract the text from the above document as if you were reading it naturally. "
    "Return the tables in html format. "
    "Return the equations in LaTeX representation. "
    "If there is an image in the document wrap it as follows <img>image here</img>. "
    "Watermarks should be wrapped in brackets. Ex: <watermark>OFFICIAL COPY</watermark>. "
    "Page numbers should be wrapped in brackets. Ex: <page_number>14</page_number>."
)


OCR_MODEL_ID = 'google/gemini-3-flash-preview'

response = raw_client.chat.completions.create(
    model= OCR_MODEL_ID,
    max_tokens=16000,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "file",
                    "file": {
                        "filename": "document.pdf",
                        "file_data": f"data:application/pdf;base64,{pdf_base64}",
                    },
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ],
)

OCR_response = response.choices[0].message.content

print("OCR done ✓")
print('-'*20)
print(OCR_response)
print('─'*80)

with open('gemini-3-flash-preview.txt', 'w') as f:
  f.write(OCR_response)

OCR done ✓
--------------------
This document presents statistical data on persons with disabilities in the Sultanate of Oman for the years 2016 and 2017, issued by the National Centre for Statistics and Information.

<page_number>6</page_number>
# Persons with Disabilities
### Community Statistics Series
### 2018 Issue

---

# Demographic Situation

---

<page_number>11</page_number>

### 1- Geographical Distribution of Persons with Disability

It is clear from Table No. (1) that the number of persons registered in the disabled person card system in the Sultanate decreased from (34,861) in 2016 to reach (34,365) disabled persons in 2017. Males constituted (65%) and females (35%) in both years. As for the governorates, the highest percentage of them was concentrated in North Al Batinah Governorate, where they constituted (20.5%) of their total at the governorate level in 2017 (68.6% male vs. 31.4% female), followed by Muscat, Ad Dakhiliyah, and the lowest in Al Wusta Governorate.

> **

In [ ]:
# @title Step 2 – Answer question with LLM-MODEL via LlamaIndex

OCR_response = open('gemini-3-flash-preview.txt', 'r').read()

LLM_ID = "google/gemini-2.5-flash"
ALL_AVAILABLE_MODELS[LLM_ID] = 1048576
CHAT_MODELS[LLM_ID] = 1048576

LLM_MODEL = OpenAI(
    model=LLM_ID,
    api_key=os.environ["OPENROUTER_API_KEY"],
    api_base="https://openrouter.ai/api/v1",
    max_tokens=16000,
)

# question = "كم عدد الذكور المعاقين من محافظة جنوب الشرقية في عام 2016"

for question in questions:
  LLM_prompt = f"""تم تزويدك بالنص المستخرج التالي من أحد المستندات:

---
{OCR_response}
---

أجب الآن عن هذا السؤال بناءً على النص المستخرج أعلاه:
{question}
"""

  messages = [
      ChatMessage(
          role=MessageRole.USER,
          blocks=[TextBlock(text=LLM_prompt)],
      )
  ]

  LLM_response = LLM_MODEL.chat(messages)
  print('─'*80)
  print(question)
  print('-'*20)
  print(LLM_response.message.content)
  print('─'*80)

────────────────────────────────────────────────────────────────────────────────
ماهي احصائيات الاشخاص المعاقين في عام 2016 ضمن محافظة الظاهرة ؟
--------------------
في عام 2016، كانت إحصائيات الأشخاص المعاقين في محافظة الظاهرة كالتالي:

*   **ذكور:** 2,240
*   **إناث:** 1,322
*   **المجموع الكلي:** 3,562
────────────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────────────
من الجدول رقم (2) ماهي الفئة العمرية التي تحتوي على اقل عدد من الاشخاص ذوي الاعاقة ؟
--------------------
من الجدول رقم (2)، الفئة العمرية التي تحتوي على أقل عدد من الأشخاص ذوي الإعاقة هي **(15 - 19)** في عام 2017، حيث بلغ عددهم 1,812 شخصًا.
────────────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────────────
من الجدول رقم (4) ماهي الاحصائيات المتعلقة بمرض المهق لسنة 2017 ؟
--------------------
من الجدول رقم (4)، الإحصائيات المتعلقة ب

# Gemini-2.5-flash

In [ ]:
# @title Step 1 – Extraction

import os
import base64
from openai import OpenAI as RawOpenAI
from llama_index.llms.openai import OpenAI
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core.base.llms.types import TextBlock
from llama_index.llms.openai.utils import ALL_AVAILABLE_MODELS, CHAT_MODELS

os.environ["OPENROUTER_API_KEY"] = 'your key'


pdf_path = "page_1.pdf"
with open(pdf_path, "rb") as f:
    pdf_base64 = base64.standard_b64encode(f.read()).decode("utf-8")


# ── Step 1 – OCR using raw OpenAI client (bypasses LlamaIndex validation) ───

raw_client = RawOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

prompt = (
    "Extract the text from the above document as if you were reading it naturally. "
    "Return the tables in html format. "
    "Return the equations in LaTeX representation. "
    "If there is an image in the document wrap it as follows <img>image here</img>. "
    "Watermarks should be wrapped in brackets. Ex: <watermark>OFFICIAL COPY</watermark>. "
    "Page numbers should be wrapped in brackets. Ex: <page_number>14</page_number>."
)


OCR_MODEL_ID =  "google/gemini-2.5-flash"

response = raw_client.chat.completions.create(
    model= OCR_MODEL_ID,
    max_tokens=16000,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "file",
                    "file": {
                        "filename": "document.pdf",
                        "file_data": f"data:application/pdf;base64,{pdf_base64}",
                    },
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ],
)

OCR_response = response.choices[0].message.content

print("OCR done ✓")
print('-'*20)
print(OCR_response)
print('─'*80)

with open('gemini-2.5-flash.txt', 'w') as f:
  f.write(OCR_response)

OCR done ✓
--------------------
<page_number>1</page_number>
<watermark>المركز الوطني
للإحصاء
و المعلومات
تعزيز المعرفة
سلطنة عُمان</watermark>
الأشخاص ذوو الإعاقة
7
سلسلة الاحصاءات المجتمعية
إصدار ٢٠١٨

<page_number>2</page_number>
الوضع الديموغرافي

<page_number>3</page_number>
<watermark>II</watermark>
1- التوزيع الجغرافي للأشخاص ذوي الإعاقة
يتضح من الجدول رقم (۱) أن عدد الأشخاص المسجلين بنظام بطاقة شخص معاق في السلطنة انخفض من
(٣٤,٨٦١) عام ۲۰۱٦ م ليصل إلى ( ٣٤,٣٦٥) معاقا في عام ۲۰۱۷م؛ شكّل الذكور ما نسبته (٦٥٪) والإناث
( ٣٥٪) ، في كلا العامين. أما بالنسبة للمحافظات فقد تركزت أعلى نسبة منهم في محافظة شمال الباطنة ، حيث
شكلوا في عام ٢٠١٧م ما نسبته (٢٠,٥%) من إجماليهم على مستوى المحافظات بواقع (٦٨,٦٪ ذكور) مقابل
(٣١,٤٪ إناث) ، ثم تليها محافظتي مسقط والداخلية وأدناها في محافظة الوسطى.
المعاق : الشخص الذي يعاني من نقص في بعض قدراته الحسية أو الجسدية أو
الذهنية خلقيا ، أو نتيجة عامل وراثي ، أو مرض ، أو حادث ، مما يحد من قدرته على
تأدية دوره الطبيعي في الحياة قياسا على من هم في عمره ، بما

In [ ]:
# @title Step 2 – Answer question with LLM-MODEL via LlamaIndex

OCR_response = open('gemini-2.5-flash.txt', 'r').read()

LLM_ID = "google/gemini-2.5-flash"
ALL_AVAILABLE_MODELS[LLM_ID] = 1048576
CHAT_MODELS[LLM_ID] = 1048576

LLM_MODEL = OpenAI(
    model=LLM_ID,
    api_key=os.environ["OPENROUTER_API_KEY"],
    api_base="https://openrouter.ai/api/v1",
    max_tokens=16000,
)

# question = "كم عدد الذكور المعاقين من محافظة جنوب الشرقية في عام 2016"

for question in questions:
  LLM_prompt = f"""تم تزويدك بالنص المستخرج التالي من أحد المستندات:

---
{OCR_response}
---

أجب الآن عن هذا السؤال بناءً على النص المستخرج أعلاه:
{question}
"""

  messages = [
      ChatMessage(
          role=MessageRole.USER,
          blocks=[TextBlock(text=LLM_prompt)],
      )
  ]

  LLM_response = LLM_MODEL.chat(messages)
  print('─'*80)
  print(question)
  print('-'*20)
  print(LLM_response.message.content)
  print('─'*80)

────────────────────────────────────────────────────────────────────────────────
ماهي احصائيات الاشخاص المعاقين في عام 2016 ضمن محافظة الظاهرة ؟
--------------------
وفقًا للجدول رقم (1) في الصفحة 3، كانت إحصائيات الأشخاص المعاقين في عام 2016 ضمن محافظة الظاهرة كالتالي:

*   **إناث:** 1,358
*   **ذكور:** 2,204
*   **الجملة:** 3,562
────────────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────────────
من الجدول رقم (2) ماهي الفئة العمرية التي تحتوي على اقل عدد من الاشخاص ذوي الاعاقة ؟
--------------------
من الجدول رقم (2)، الفئة العمرية التي تحتوي على أقل عدد من الأشخاص ذوي الإعاقة هي **(15 - 19)** في عام 2017 بعدد 1,812 شخص.
────────────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────────────
من الجدول رقم (4) ماهي الاحصائيات المتعلقة بمرض المهق لسنة 2017 ؟
--------------------
من الجدول رقم (4) لسنة 20

# Baseer

In [ ]:
# @title Step 1 – Extraction

# !pip install kawn.ai==0.1.1

from kawn import KawnClient
from kawn.services import OCRService
import os
import base64
from openai import OpenAI as RawOpenAI
from llama_index.llms.openai import OpenAI
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core.base.llms.types import TextBlock
from llama_index.llms.openai.utils import ALL_AVAILABLE_MODELS, CHAT_MODELS

os.environ["OPENROUTER_API_KEY"] = 'your key'

os.environ["MISRAJ_API_KEY"] = "your key"

# The client automatically picks up the MISRAJ_API_KEY environment variable.
client = KawnClient()

# Initialize our decoupled services using the core client transport
ocr_service = OCRService(client)

# Basser OCR: Single File Processing (Supports Images and PDFs)
OCR_response = ocr_service.process_file(file_path="page_1.pdf")

pages_ocr =''
for page in OCR_response.pages:
  pages_ocr = pages_ocr + page.content + '\n'
OCR_response = pages_ocr

print("OCR done ✓")
print('-'*20)
print(OCR_response)
print('─'*80)


with open('Baseer.txt', 'w') as f:
  f.write(OCR_response)

[Kawn][Rank 0] 2026-04-14 07:57:23,883 - [OCR Service]-0 - INFO - OCR request has correctly upload the file to the server and receive task job id: 73dec6a8-fe0a-4887-b36e-8d129ede1a7b
INFO:[OCR Service]-0:OCR request has correctly upload the file to the server and receive task job id: 73dec6a8-fe0a-4887-b36e-8d129ede1a7b
[Kawn][Rank 0] 2026-04-14 07:57:23,888 - [OCR Service]-0 - INFO - Waiting the response...
INFO:[OCR Service]-0:Waiting the response...
[Kawn][Rank 0] 2026-04-14 08:00:19,545 - [OCR Service]-0 - INFO - The processing done successfully, we will get the result.
INFO:[OCR Service]-0:The processing done successfully, we will get the result.
[Kawn][Rank 0] 2026-04-14 08:00:19,550 - [OCR Service]-0 - WARNING - Please save the result, as it will be deleted as long as you requested from the server.


OCR done ✓
--------------------
[IMAGE]

المؤتمر العالمي للإحصاء

2008

\**سلسلة الاحصاءات المجتمعية**
\**إصدار ٢٠١٨**

&lt;page_number&gt;٦&lt;/page_number&gt;
\**الأشخاص ذوو الإعاقة**
\**الوضع الديموغرافي**
&lt;page_number&gt;||&lt;/page_number&gt;

\**١- التوزيع الجغرافي للأشخاص ذوي الإعاقة**

يتضح من الجدول رقم (١) أن عدد الأشخاص المسجلين بنظام بطاقة شخص معاق في السلطنة انخفض من (٣٤,٨٦١) عام ٢٠١٦م ليصل إلى ( ٣٤,٣٦٥) معاقاً في عام ٢٠١٧م؛ شكل الذكور ما نسبته (٦٥٪) والإناث (٣٥٪)، في كلا العامين. أما بالنسبة للمحافظات فقد تركزت أعلى نسبة منهم في محافظة شمال الباطنة ، حيث شكلوا في عام ٢٠١٧م ما نسبته (٢٠,٥٪) من إجماليهم على مستوى المحافظات بواقع (٦٨,٦٪ ذكور) مقابل (٣١,٤٪ إناث) ، ثم تليها محافظتي مسقط والداخلية وأدناها في محافظة الوسطى.

\**المعاق:** الشخص الذي يعاني من نقص في بعض قدراته الحسية أو الجسدية أو الذهنية خلقياً ، أو نتيجة عامل وراثي ، أو مرض ، أو حادث ، مما يحد من قدرته على تأدية دوره الطبيعي في الحياة قياساً على من هم في عمره ، بما يحتاج معه إلى الرعاية والتأهيل حتى يؤدى دوره

In [ ]:
# @title Step 2 – Answer question with LLM-MODEL via LlamaIndex

OCR_response = open('Baseer.txt', 'r').read()

LLM_ID = "google/gemini-2.5-flash"
ALL_AVAILABLE_MODELS[LLM_ID] = 1048576
CHAT_MODELS[LLM_ID] = 1048576

LLM_MODEL = OpenAI(
    model=LLM_ID,
    api_key=os.environ["OPENROUTER_API_KEY"],
    api_base="https://openrouter.ai/api/v1",
    max_tokens=16000,
)

# question = "كم عدد الذكور المعاقين من محافظة جنوب الشرقية في عام 2016"

for question in questions:
  LLM_prompt = f"""تم تزويدك بالنص المستخرج التالي من أحد المستندات:

---
{OCR_response}
---

أجب الآن عن هذا السؤال بناءً على النص المستخرج أعلاه:
{question}
"""

  messages = [
      ChatMessage(
          role=MessageRole.USER,
          blocks=[TextBlock(text=LLM_prompt)],
      )
  ]

  LLM_response = LLM_MODEL.chat(messages)
  print('─'*80)
  print(question)
  print('-'*20)
  print(LLM_response.message.content)
  print('─'*80)

────────────────────────────────────────────────────────────────────────────────
ماهي احصائيات الاشخاص المعاقين في عام 2016 ضمن محافظة الظاهرة ؟
--------------------
في عام 2016، بلغ عدد الأشخاص المعاقين في محافظة الظاهرة 3562 شخصًا، منهم 1358 إناث و 2240 ذكور.
────────────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────────────
من الجدول رقم (2) ماهي الفئة العمرية التي تحتوي على اقل عدد من الاشخاص ذوي الاعاقة ؟
--------------------
من الجدول رقم (2)، الفئة العمرية التي تحتوي على أقل عدد من الأشخاص ذوي الإعاقة هي **(15 - 19)** في عام 2017 بعدد 1,812 شخصًا.
────────────────────────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────────────────────────
من الجدول رقم (4) ماهي الاحصائيات المتعلقة بمرض المهق لسنة 2017 ؟
--------------------
من الجدول رقم (4) لسنة 2017، الإحصائيات المتعلقة بمرض المهق هي:

*   **المجموع:** 5
*   **(أكثر